# Job-Laufzeiten AS/400 – Erkundungs-Notebook

**Ziel:** Start- und Endzeit von Jobs mit Namen `%TAGEND_NOP%` finden.

**Sicherheit:** Alle Schritte sind reine Lesezugriffe. Kein Schreiben außer in das eigene Schema. Keine Loops. Jeder Block ist unabhängig – ein Fehler stoppt nur diesen Block, nicht das System.

**Reihenfolge:** Einfach von oben nach unten ausführen. Bei jedem Block steht was er tut und ob er geklappt hat.

In [ ]:
# ── Konfiguration – hier anpassen ────────────────────────────────────────────
MY_SCHEMA  = 'STADTHERR'
JOB_FILTER = '%TAGEND_NOP%'

# Datum für Tests: nur EINEN Tag abfragen – minimale Last!
# Format: JJMMTT (also z.B. 260414 = 14. April 2026)
TEST_DATUM = '260414'   # <-- gestern oder heute eintragen

# ── Verbindung ───────────────────────────────────────────────────────────────
from poweriodbc import PoweriODBC
import pandas as pd

cnxn = PoweriODBC().connect()
crsr = cnxn.cursor()
print('Verbindung OK')

---
## WEG A – DSPLOG → Spool → CPYSPLF → Text parsen
DSPLOG unterstützt kein OUTPUT(*OUTFILE), aber OUTPUT(*PRINT) erzeugt eine Spool-Datei.
Diese kopieren wir dann als Text in unsere Bibliothek und parsen sie.

In [ ]:
# ── Weg A, Schritt 1: DSPLOG für NUR EINEN TAG → Spool ───────────────────────
# PERIOD begrenzt auf einen Tag → minimale Ausgabe, keine Last!
# OUTPUT(*PRINT) ist rein lesend, erzeugt nur eine Spool-Datei für dich.

try:
    cmd = f"DSPLOG LOG(QHST) OUTPUT(*PRINT) PERIOD(('000000' '{TEST_DATUM}') ('235959' '{TEST_DATUM}'))"
    crsr.execute(f"CALL QSYS2.QCMDEXC('{cmd}', {len(cmd):.5f})")
    print('✅ Weg A Schritt 1 OK: Spool-Datei wurde erstellt')
except Exception as e:
    print(f'❌ Weg A Schritt 1 fehlgeschlagen: {e}')

In [ ]:
# ── Weg A, Schritt 2: Spool-Datei in eigene Bibliothek kopieren ──────────────
# CPYSPLF kopiert die zuletzt erstellte Spool-Datei als Text-Datei.
# CRTFILE(*YES) erstellt die Zieltabelle automatisch falls nicht vorhanden.

try:
    cmd = f"CPYSPLF FILE(QPJOBLOG) TOFILE({MY_SCHEMA}/QHST_SPOOL) SPLNBR(*LAST) CRTFILE(*YES) MBROPT(*REPLACE)"
    crsr.execute(f"CALL QSYS2.QCMDEXC('{cmd}', {len(cmd):.5f})")
    print('✅ Weg A Schritt 2 OK: Spool kopiert nach', MY_SCHEMA, '.QHST_SPOOL')
except Exception as e:
    print(f'❌ Weg A Schritt 2 fehlgeschlagen: {e}')
    print('   → Tipp: Spool-Dateiname könnte anders heißen (s. nächste Zelle)')

In [ ]:
# ── Weg A, Schritt 2b: Falls Schritt 2 fehlschlug – Spool-Dateiname ermitteln
# Zeigt deine letzten 5 Spool-Dateien – so sehen wir den richtigen Namen.

try:
    cmd = f"WRKSPLF SELECT(*ALL) OUTPUT(*OUTFILE) OUTFILE({MY_SCHEMA}/SPOOL_LIST) OUTMBR(*FIRST *REPLACE)"
    crsr.execute(f"CALL QSYS2.QCMDEXC('{cmd}', {len(cmd):.5f})")
    df_spl = pd.read_sql(f"SELECT * FROM {MY_SCHEMA}.SPOOL_LIST ORDER BY SPLCDT DESC FETCH FIRST 5 ROWS ONLY", cnxn)
    print('Letzte Spool-Dateien:')
    display(df_spl)
except Exception as e:
    print(f'❌ WRKSPLF fehlgeschlagen: {e}')

In [ ]:
# ── Weg A, Schritt 3: Spool-Inhalt anzeigen und nach TAGEND_NOP suchen ────────

try:
    # Erste 20 Zeilen zur Strukturprüfung
    df_spool = pd.read_sql(f"SELECT * FROM {MY_SCHEMA}.QHST_SPOOL FETCH FIRST 20 ROWS ONLY", cnxn)
    print('Struktur der Spool-Datei:')
    display(df_spool)
except Exception as e:
    print(f'❌ Spool-Tabelle nicht lesbar: {e}')

In [ ]:
# ── Weg A, Schritt 4: Nach CPF1164 / TAGEND_NOP filtern ──────────────────────
# Spaltennamen ggf. aus Schritt 3 anpassen!
# Typischer Spaltenname für den Zeileninhalt: SPLFDATA oder QPRTLINE o.ä.

try:
    # Zuerst Spaltenname der Textzeile ermitteln
    cols = pd.read_sql(f"SELECT COLUMN_NAME FROM QSYS2.SYSCOLUMNS WHERE TABLE_SCHEMA='{MY_SCHEMA}' AND TABLE_NAME='QHST_SPOOL'", cnxn)
    print('Spalten:', cols['COLUMN_NAME'].tolist())

    # Häufigster Spaltenname ist QPRTLINE – ggf. anpassen
    text_col = 'QPRTLINE'
    df_hits = pd.read_sql(
        f"SELECT * FROM {MY_SCHEMA}.QHST_SPOOL WHERE {text_col} LIKE '%TAGEND_NOP%' OR {text_col} LIKE '%CPF1164%'",
        cnxn
    )
    print(f'✅ {len(df_hits)} Treffer:')
    display(df_hits)
except Exception as e:
    print(f'❌ Filter fehlgeschlagen: {e}')

---
## WEG B – QACGJRN direkt per SQL (Table Function)
Kein CL-Befehl, kein OUTPUT(*OUTFILE) – direkt per SQL auf das Journal zugreifen.
Funktioniert nur wenn Job-Accounting aktiviert ist.

In [ ]:
# ── Weg B, Schritt 1: Ist Job-Accounting aktiv? ──────────────────────────────

try:
    df_acg = pd.read_sql(
        "SELECT SYSTEM_VALUE_NAME, CURRENT_NUMERIC_VALUE, CURRENT_CHARACTER_VALUE "
        "FROM QSYS2.SYSTEM_VALUE_INFO WHERE SYSTEM_VALUE_NAME = 'QACGLVL'",
        cnxn
    )
    print('QACGLVL (Job-Accounting):')
    display(df_acg)
    print('(*NONE = aus, *JOB oder *JOBDTA = aktiv)')
except Exception as e:
    print(f'❌ Systemwert nicht lesbar: {e}')

In [ ]:
# ── Weg B, Schritt 2: DISPLAY_JOURNAL Table Function ─────────────────────────
# ⚠️ FETCH FIRST 10 ROWS ONLY – wichtig! Journal kann sehr groß sein.

try:
    df_jrn = pd.read_sql(
        """
        SELECT *
        FROM TABLE(QSYS2.DISPLAY_JOURNAL(
            JOURNAL_LIBRARY => 'QSYS',
            JOURNAL_NAME    => 'QACGJRN'
        )) AS X
        FETCH FIRST 10 ROWS ONLY
        """,
        cnxn
    )
    print(f'✅ DISPLAY_JOURNAL funktioniert! {len(df_jrn)} Zeilen')
    display(df_jrn)
except Exception as e:
    print(f'❌ DISPLAY_JOURNAL nicht verfügbar (pre-V7 oder nicht berechtigt): {e}')

In [ ]:
# ── Weg B, Schritt 3: Falls Schritt 2 OK – nach TAGEND_NOP filtern ────────────
# ⚠️ Nur ausführen wenn Schritt 2 funktioniert hat!

try:
    df_jrn_jobs = pd.read_sql(
        f"""
        SELECT *
        FROM TABLE(QSYS2.DISPLAY_JOURNAL(
            JOURNAL_LIBRARY => 'QSYS',
            JOURNAL_NAME    => 'QACGJRN'
        )) AS X
        WHERE CAST(ENTRY_DATA AS VARCHAR(1000)) LIKE '{JOB_FILTER}'
        FETCH FIRST 20 ROWS ONLY
        """,
        cnxn
    )
    print(f'✅ {len(df_jrn_jobs)} Treffer für {JOB_FILTER}')
    display(df_jrn_jobs)
except Exception as e:
    print(f'❌ Filter fehlgeschlagen: {e}')

---
## WEG C – QSYS2 Table Functions (SQL, kein CL)
Verschiedene moderne QSYS2-Views direkt per SQL testen.

In [ ]:
# ── Weg C: Mehrere QSYS2-Views auf einen Schlag testen ───────────────────────
# Einfach alle durchprobieren – Fehler werden abgefangen.

kandidaten = [
    ("QSYS2.JOBLOG_INFO",           f"SELECT * FROM QSYS2.JOBLOG_INFO WHERE JOB_NAME LIKE '{JOB_FILTER}' FETCH FIRST 5 ROWS ONLY"),
    ("QSYS2.JOB_INFO",              f"SELECT * FROM TABLE(QSYS2.JOB_INFO(JOB_USER_FILTER=>'*ALL')) AS X WHERE JOB_NAME LIKE '{JOB_FILTER}' FETCH FIRST 5 ROWS ONLY"),
    ("QSYS2.OUTPUT_QUEUE_ENTRIES",  f"SELECT * FROM QSYS2.OUTPUT_QUEUE_ENTRIES WHERE JOB_NAME LIKE '{JOB_FILTER}' FETCH FIRST 5 ROWS ONLY"),
    ("QSYS2.SCHEDULED_JOB_INFO",    f"SELECT * FROM QSYS2.SCHEDULED_JOB_INFO FETCH FIRST 5 ROWS ONLY"),
    ("QSYS2.SYSPROGRAMSTAT",        f"SELECT * FROM QSYS2.SYSPROGRAMSTAT FETCH FIRST 5 ROWS ONLY"),
]

for name, sql in kandidaten:
    try:
        df = pd.read_sql(sql, cnxn)
        print(f'✅ {name}: {len(df)} Zeilen gefunden')
    except Exception as e:
        print(f'❌ {name}: nicht verfügbar')

---
## WEG D – WRKSPLF direkt per SQL
Spool-Dateien abfragen ohne CL-Befehl.

In [ ]:
# ── Weg D: Spool-Einträge per SQL ────────────────────────────────────────────

spool_kandidaten = [
    "SELECT * FROM QSYS2.OUTPUT_QUEUE_ENTRIES FETCH FIRST 10 ROWS ONLY",
    "SELECT * FROM QUSRSYS.QSPLJOB FETCH FIRST 10 ROWS ONLY",
    "SELECT * FROM QSYS.QSPLJOB FETCH FIRST 10 ROWS ONLY",
]

for sql in spool_kandidaten:
    try:
        df = pd.read_sql(sql, cnxn)
        print(f'✅ Funktioniert: {sql[:60]}...')
        display(df.head(3))
    except Exception as e:
        print(f'❌ {sql[:60]}: nicht verfügbar')

---
## WEG E – QHST Members anzeigen
Wie weit reicht die QHST-Historie? Und welche Members gibt es?

In [ ]:
# ── Weg E: QHST-Members anzeigen ─────────────────────────────────────────────
# DSPFD zeigt Datei-Beschreibung inkl. aller Members

try:
    cmd = f"DSPFD FILE(QSYS/QHST) TYPE(*MBR) OUTPUT(*OUTFILE) OUTFILE({MY_SCHEMA}/QHST_MBR) OUTMBR(*FIRST *REPLACE)"
    crsr.execute(f"CALL QSYS2.QCMDEXC('{cmd}', {len(cmd):.5f})")
    df_mbr = pd.read_sql(f"SELECT * FROM {MY_SCHEMA}.QHST_MBR", cnxn)
    print(f'✅ QHST hat {len(df_mbr)} Members (= Tage/Perioden):')
    display(df_mbr)
except Exception as e:
    print(f'❌ DSPFD fehlgeschlagen: {e}')

In [ ]:
# ── Alternativ: DSPLOG nur für einen Tag mit PERIOD ──────────────────────────
# Zeigt wie weit die Historie reicht – einfach Datum variieren
# Datum-Format: JJMMTT

for datum in [TEST_DATUM, '260413', '260412', '260411']:
    try:
        cmd = f"DSPLOG LOG(QHST) OUTPUT(*PRINT) PERIOD(('000000' '{datum}') ('235959' '{datum}'))"
        crsr.execute(f"CALL QSYS2.QCMDEXC('{cmd}', {len(cmd):.5f})")
        print(f'✅ Datum {datum}: Daten vorhanden')
    except Exception as e:
        print(f'❌ Datum {datum}: keine Daten oder Fehler')

In [ ]:
# ── Abschluss ─────────────────────────────────────────────────────────────────
crsr.close()
cnxn.close()
print('Fertig. Bitte Ergebnisse notieren und mit Stefan besprechen.')